# What is the Identity Key and how should it work? 


**FOCUS**: Where are the gaps in our current idenity process? 

In the new solution, we know that in the sales, GlobalId(GI) is a linked to the SP customer. SPCustomerKey(SP) is assoicated with the Pass and ATCustomerKey(AT) is associated with the buyer.

SP -> GD is 1:1\
GD -> SP is 1:M

The reason we know this is due to previous research.\
GDs that have more than on SP is due to that SP has bought at more than one location. So although its 1:M, the person is the same in all accounts. 

What we are going to look at is the hole that are in the current solution and where they fall apart. By doing that we should be able to adjust and fix for the new solution


Those CustomerKeys that are null, this is due mostly to products that are bought at the park. There is nothing much we can do about that (most of ticket year 2026 are OT02)

## Conclusion

1. Indiv_keys are missing due fallback not working (not looking for emails that have an indiv_key and assign the missing links)
2. -1s should no longer be an issue with the new solution with only less than 
3. HH logic should be looked at (Lastname + Addresskey) is not spliting properly (Muliple last names in the same household)
    
    3.1 Majority of household sizes are ragned from 1 to 5
    
4. Additional insights on null/space counts in the DIM SP/AT customer tables.

## Looking at current solution and seeing where things break

In the current solution the `CRM Customer No` is built by this foundation from the FACT_SALES table

```
    IF SPKEY LIKE '%NONE%' THEN ATKEY ELSE SPKEY 
```


we can prove this logic by looking at the EXT_CUSTOMER_LINK in VW_PARK_CUSTOMER and compare it to either the SPKey(either the sales table directly or the DIM_SPCustomer) or the ATNAME(located in DIM_ATCUSTOMER)

For the Indiv_Key which is Merkel the logic is also simple yet logcial.

```
INDIVIDUAL - 1. First name & Last Name OR whole name
                2. Email OR Address
                    2.5 Address must contain:
                        - Address1 WITH (City & STATE) OR Postal_Code.
HOUSEHOLD - Last name and address grouping
ADDRESS - Unique mailing or physical address

```

Failure to have this would lead to no Indiv_key. HOWEVER in `SIXFL_647` we have a logic for Email & Phone only where if they do not have an indiv_key to look up email or phone if individual does and assign it that indiv_key. 

***

Now the issue here is we do not check to see if:\
A. SPkey when looked at first fills all of these requirements
B. When SPkey fails to meet all requirements we have no back-up.

Look below for example 

In [ ]:
%%sql -r dataframe_2
/*
===================================================================================================================
Why are there missing indiv_keys when this join is done?
====================================================================================================================
*/

--- Does VW_INDIV_PARK_PREF (park specific ops table) have the same opt_out sources?



WITH 

-- 1) Linking salesa table for the proper year with individual keys
main_customers AS (
SELECT
    pc.indiv_key,
    pr.park_id,
    pr.agc_id,
    --pr.agsc_id,
    --pr.product_name,
    pr.venue,
    pr.crm_customer_no,
    pr.order_detail_id,
    pr.global_id,
    SHA2(pc.first_name,256) as first_name,
    SHA2(pc.last_name,256) as last_name,
    SHA2(pc.email_address,256) as email_address,
    pc.ext_customer_link_id
FROM SIXFL_PROD.CONSUMPTION.VW_PARK_SALES as pr
JOIN SIXFL_PROD.CONSUMPTION.VW_PARK_CUSTOMER as pc
    ON pr.crm_customer_no = pc.crm_customer_no
    AND pr.park_id = pc.park_id
WHERE pr.agc_id LIKE ('SP%') 
    AND pr.ticket_year = 2026
    AND is_admissions = 1
    --AND pr.venue != 'WPARK' --- Ignores water park
)

SELECT *
FROM main_customers
WHERE indiv_key IS NULL
AND global_id = 'CW142495199' -- selecting individual 
;

In [ ]:
%%sql -r dataframe_5
/*
===================================================================================================================
Why are there missing indiv_keys when this join is done?
====================================================================================================================
*/

--- Does VW_INDIV_PARK_PREF (park specific ops table) have the same opt_out sources?



-- 1) Linking salesa table for the proper year with individual keys

SELECT
    pc.indiv_key,
    pr.park_id,
    pr.agc_id,
    pc.ext_customer_link_id,
    --pr.agsc_id,
    --pr.product_name,
    pr.venue,
    pr.crm_customer_no,
    pr.order_detail_id,
    pr.global_id,
    SHA2(pc.first_name,256) as first_name,
    SHA2(pc.last_name,256) as last_name,
    pc.email_address,
    pc.address_line1,
    pc.city,
    pc.postal_cd,
    pc.email_key
FROM SIXFL_PROD.CONSUMPTION.VW_PARK_SALES as pr
JOIN SIXFL_PROD.CONSUMPTION.VW_PARK_CUSTOMER as pc
    ON pr.crm_customer_no = pc.crm_customer_no
    AND pr.park_id = pc.park_id
WHERE 1=1
    --AND pr.agc_id LIKE ('SP%') 
    --AND pr.ticket_year = 2026
    --AND is_admissions = 1
    AND pc.ext_customer_link_id = 'W40100757265'
    OR (pc.crm_customer_no = '102408909'
    AND pc.park_id = 'CW')

    
;

In our first block we see [REDACTED] who only has a last name, email_address (they do have an address but removed for documenting)\
Since they do not match Merkels requirements they are not given an Indiv_key.

When comparing with the 2nd block and the 3 block(under this text) we can see the proof that the ATName and SPkey are the link between tables.\
With that, in the current solution (block 2) when we pull those Ext_Customer_link_Ids, we find an indiv_key related to an individual with similar information. 

This individual is the purchaser.

In [ ]:
%%sql -r dataframe_3
SELECT b."AttendanceGroupCategoryName",
SHA2(TRIM(UPPER(sp."FirstName")), 256) AS sp_firstname,
SHA2(TRIM(UPPER(sp."LastName")), 256) AS sp_lastname,
SHA2(TRIM(UPPER(att."FirstName")), 256) AS at_lastname,
SHA2(TRIM(UPPER(att."LastName")), 256) AS at_lastname,
att."ATCustomerName" ,
SHA2(TRIM(UPPER(sp."EmailAddress")), 256) AS sp_email,
SHA2(TRIM(UPPER(att."EmailAddress")), 256) AS at_email,
SHA2(TRIM(UPPER(sp."Phone")), 256) AS sp_ph,
SHA2(TRIM(UPPER(att."Phone")), 256) AS at_ph,
SHA2(TRIM(UPPER(sp."Address1")), 256) AS "Address1",
SHA2(TRIM(UPPER(att."BillingAddress1")), 256) AS "BillingAddress1",
a."SPCustomerKey",
a."ATCustomerKey",
a."OrderDetailName",
a."UpgradedFromTicketId",
a."TicketId",
sp."Barcode",
sp."GlobalId"
FROM ORGDATACLOUD$INTERNAL$PROD_CONSUMPTION_CDP_LISTING.CDP.CDP_FACT_SALES a 
LEFT JOIN ORGDATACLOUD$INTERNAL$PROD_CONSUMPTION_CDP_LISTING.CDP.CDP_DIM_PRODUCTS b ON a."ProductKey" = b."ProductKey"
LEFT JOIN ORGDATACLOUD$INTERNAL$PROD_CONSUMPTION_CDP_LISTING.CDP.CDP_DIM_SPCUSTOMERS sp ON a."SPCustomerKey" = sp."SPCustomerKey"
LEFT JOIN ORGDATACLOUD$INTERNAL$PROD_CONSUMPTION_CDP_LISTING.CDP.CDP_DIM_ATCUSTOMERS att ON a."ATCustomerKey" = att."ATCustomerKey"
WHERE a."GlobalId" = 'CW142495199'

To dive deeper into that we need to see how the table were populated to begin with.\
in Block 3, we pulled all information of said individual that was missing their indiv_key.\
Here we can see that in the data itself, it is missing the first name with a `space`, not even a null!
But we can see that they one of them is filled with their ATKEY information which matches to the 3rd row in block 2.\

For row 2 in block 3 that has no ATkey information, we can look and see that this purchase was not a purchase but an upgrade.\
Now with handling that we can look at it in two ways, if SP values are not null, then good. we can ignore. If they are bad, we would need to pull the ATKey from that upgraded value.

```
LOGIC :
---------------
    SELECT
        CASE
            WHEN sale."SPCustomerKey" LIKE '%None%' THEN
                COALESCE(
                    CASE
                        WHEN b."ATCustomerKey" IS NULL
                             OR b."ATCustomerKey" LIKE '%None%'
                        THEN NULL
                        ELSE b."ATCustomerKey"
                    END,
                    CASE
                        WHEN sale."ATCustomerKey" IS NULL
                             OR sale."ATCustomerKey" LIKE '%None%'
                        THEN NULL
                        ELSE sale."ATCustomerKey"
                    END
                )
            ELSE sale."SPCustomerKey"
        END AS "CustomerKey",
    FROM ORGDATACLOUD$INTERNAL$PROD_CONSUMPTION_CDP_LISTING.CDP.CDP_FACT_SALES sale

    LEFT JOIN ORGDATACLOUD$INTERNAL$PROD_CONSUMPTION_CDP_LISTING.CDP.CDP_FACT_SALES b
        ON sale."UpgradedFromTicketId" = b."TicketId"

```

By applying this we can have a back up incase the SPKey leads to any errors during the identification process.

**After speaking with [REDACTED] and showing her this example, we found 22,691 rows in the `VW_PARK_CUSTOMER` that had a missing indiv_key. [REDACTED] agreed that the those with an email should have been assigned an Indiv_key and that this will be fixed in the new solution.**


## When CRMCustomerNO is null or '-1' in current solution
---

In [ ]:
%%sql -r dataframe_4
WITH CURRENTSOLUTION as (
SELECT DISTINCT -- 1,506,544 DISTINCT order_detail_id rows with a -1 in Current solution
crm_customer_no,
order_detail_id,
FROM SIXFL_PROD.CONSUMPTION.VW_PARK_SALES
WHERE TICKET_YEAR = 2026
AND (CRM_CUSTOMER_NO = -1
OR CRM_CUSTOMER_NO IS NULL )
)

SELECT DISTINCT -- when looking, we have 3,131 rows that do not match, 
COALESCE(a."OrderDetailName",CONCAT("SiteKey",'~',"SalesTransactionId")) as order_id,
cs.order_detail_id
FROM ORGDATACLOUD$INTERNAL$PROD_CONSUMPTION_CDP_LISTING.CDP.CDP_FACT_SALES a 
RIGHT JOIN CURRENTSOLUTION cs ON COALESCE(a."OrderDetailName",CONCAT("SiteKey",'~',"SalesTransactionId")) = cs.order_detail_id 

in the Current solution there 1,506,544

When looking at the difference between the current and new solution to see if they have the same order_ids, we see that in the current solution when an order_id is not populated it uses a concat of site_key + salesTransactionID to make an order_id. this only happens when there is an upgrade in the new solution which impacts the current solution layout. 

Granted, its only 3,131 rows that do not have a match when looking at the two solution but those are being looked into by IT

In [ ]:
%%sql -r dataframe_6
WITH CS as (
SELECT DISTINCT -- 1,506,544 DISTINCT order_detail_id rows with a -1 in Current solution
crm_customer_no,
order_detail_id,
FROM SIXFL_PROD.CONSUMPTION.VW_PARK_SALES
WHERE TICKET_YEAR = 2026
AND (CRM_CUSTOMER_NO = -1
OR CRM_CUSTOMER_NO IS NULL )
),

NS as (
SELECT DISTINCT  -- matched 99%
a."ATCustomerKey",
a."SPCustomerKey", 
COALESCE(a."OrderDetailName",CONCAT("SiteKey",'~',"SalesTransactionId")) as order_id,
CASE WHEN a."OrderDetailName" IS NULL THEN 1 ELSE 0 END AS upgraded,
cs.order_detail_id,
a."ProductKey"
FROM ORGDATACLOUD$INTERNAL$PROD_CONSUMPTION_CDP_LISTING.CDP.CDP_FACT_SALES a 
RIGHT JOIN  CS ON COALESCE(a."OrderDetailName",CONCAT("SiteKey",'~',"SalesTransactionId")) = cs.order_detail_id 
),

validtype as (
SELECT b."AttendanceGroupCategoryName", -- added about 500K due to multiple individuals 
SHA2(TRIM(UPPER(sp."FirstName")), 256) AS sp_firstname,
SHA2(TRIM(UPPER(sp."LastName")), 256) AS sp_lastname,
SHA2(TRIM(UPPER(att."FirstName")), 256) AS at_firstname,
SHA2(TRIM(UPPER(att."LastName")), 256) AS at_lastname,
att."ATCustomerName",
CASE WHEN sp."FirstName" IS NULL OR sp."LastName" IS NULL  THEN 1 ELSE 0 END AS sp_no_valid,
CASE WHEN att."FirstName" IS NULL OR att."LastName" IS NULL  THEN 1 ELSE 0 END AS at_no_valid,
a.*
FROM NS a 
LEFT JOIN ORGDATACLOUD$INTERNAL$PROD_CONSUMPTION_CDP_LISTING.CDP.CDP_DIM_PRODUCTS b ON a."ProductKey" = b."ProductKey"
LEFT JOIN ORGDATACLOUD$INTERNAL$PROD_CONSUMPTION_CDP_LISTING.CDP.CDP_DIM_SPCUSTOMERS sp ON a."SPCustomerKey" = sp."SPCustomerKey"
LEFT JOIN ORGDATACLOUD$INTERNAL$PROD_CONSUMPTION_CDP_LISTING.CDP.CDP_DIM_ATCUSTOMERS att ON a."ATCustomerKey" = att."ATCustomerKey"
WHERE  a.order_id is not null 
),

invalid as (
SELECT DISTINCT -- 2,002,902 rows when merging the sp and at customer tables
CASE WHEN sp_no_valid = 1 AND at_no_valid = 1 THEN 1 ELSE 0 END as invalid 
,* exclude(sp_no_valid,at_no_valid)
FROM validtype
)

SELECT *
FROM invalid
--WHERE invalid = 1

This block if focusing on either SPkey or ATkey are able to fill one of the -1s found in the current solution. the results of this block is those that are not valid for an indiv key.\
**Besides the only 35 rows that are not able to find valid Indiv keys,\
we can assume that -1s should be near gone in the new solution based on this analysis.**

## Household IDs
---

```
LOGIC:
------
HOUSEHOLD - Last name and address grouping
```
This will be the first test we will check in the indiv_summary table on the current solution.

In [ ]:
%%sql -r dataframe_7
SELECT
best_cr_household_key,
count(distinct last_name),
count(distinct best_cr_addr_key),
count(distinct best_email_key)
FROM sixfl_prod.consumption.vw_indiv_summary
WHERE best_cr_household_key IS NOT NULL
AND best_cr_household_key <> '0' 
GROUP BY ALL 
having count(distinct last_name) > 1


While looking at those households (HH), we can see that we have 15,351,057 Distinct keys.\
Those that have more than one last name is only 174,286 HH keys.\
However, of those 174K all have 1 distinct address_key assigned to them.

**We might want to apply a more conservative rule with the address key**

In [ ]:
%%sql -r dataframe_8
WITH base as (
SELECT best_cr_household_key,count(*) as counts
FROM SIXFL_PROD.CONSUMPTION.VW_INDIV_SUMMARY
WHERE best_cr_household_key <> '0'
AND best_cr_household_key IS NOT NULL 
GROUP BY ALL 

)

SELECT 
counts as "Number of individuals in a household",
count(*) as "ttl count"
FROM base
GROUP BY ALL 
ORDER BY counts asc 
--WHERE best_cr_household_key = '52VT1YY6SBV3J3LCJJZC5W8NAT'

When examining the number of individuals associated with a single household (HH), we found that 99% of households have between 1 and 5 members. Merkel noted that the maximum household size can be as high as 93. However, given the known gaps in the household assignment logic, these household size distributions may change as the logic is refined.

## Additional findings

Here we are looking at values that are `blank` or `null`. This can help identify which columns would be good for creating the idenity process.

In [ ]:
# --- 1. Imports ---
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col, count, when, lit, trim
from functools import reduce

# --- 2. Session ---
session = get_active_session()

# --- 3. Load table ---
df = session.table(
    "ORGDATACLOUD$INTERNAL$PROD_CONSUMPTION_CDP_LISTING.CDP.CDP_DIM_ATCUSTOMERS"
)

# --- 4. Get schema (to detect column types) ---
schema = {field.name: field.datatype for field in df.schema.fields}

# --- 5. Total rows ---
total_rows = df.count()

# --- 6. Build logic safely ---
dfs = []

for c in df.columns:
    
    col_type = str(schema[c]).lower()

    # ✅ Always include NULL check
    condition = col(c).is_null()

    # ✅ Only apply blank/space logic for string columns
    if "string" in col_type:
        condition = condition | (trim(col(c)) == '')

    dfs.append(
        df.select(
            lit(c).alias("COLUMN_NAME"),
            count(when(condition, c)).alias("MISSING_COUNT"),
            (count(when(condition, c)) / total_rows).alias("MISSING_PCT")
        )
    )

# Combine all
result = reduce(lambda x, y: x.union_all(y), dfs)

# --- 7. Convert to Pandas (safe) ---
pdf = result.to_pandas()

# --- 8. Format ---
pdf["MISSING_COUNT"] = pdf["MISSING_COUNT"].map("{:,}".format)
pdf["MISSING_PCT"] = (pdf["MISSING_PCT"] * 100).round(2).astype(str) + "%"

# --- 9. Display ---
pdf

In [ ]:
# --- 1. Imports ---
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col, count, when, lit, trim
from functools import reduce

# --- 2. Session ---
session = get_active_session()

# --- 3. Load table ---
df = session.table(
    "ORGDATACLOUD$INTERNAL$PROD_CONSUMPTION_CDP_LISTING.CDP.CDP_DIM_SPCUSTOMERS"
)

# --- 4. Get schema (to detect column types) ---
schema = {field.name: field.datatype for field in df.schema.fields}

# --- 5. Total rows ---
total_rows = df.count()

# --- 6. Build logic safely ---
dfs = []

for c in df.columns:
    
    col_type = str(schema[c]).lower()

    # ✅ Always include NULL check
    condition = col(c).is_null()

    # ✅ Only apply blank/space logic for string columns
    if "string" in col_type:
        condition = condition | (trim(col(c)) == '')

    dfs.append(
        df.select(
            lit(c).alias("COLUMN_NAME"),
            count(when(condition, c)).alias("MISSING_COUNT"),
            (count(when(condition, c)) / total_rows).alias("MISSING_PCT")
        )
    )

# Combine all
result = reduce(lambda x, y: x.union_all(y), dfs)

# --- 7. Convert to Pandas (safe) ---
pdf = result.to_pandas()

# --- 8. Format ---
pdf["MISSING_COUNT"] = pdf["MISSING_COUNT"].map("{:,}".format)
pdf["MISSING_PCT"] = (pdf["MISSING_PCT"] * 100).round(2).astype(str) + "%"

# --- 9. Display ---
pdf

When comparing the **Missing_PCT** between the SP and AT customer tables, we can see that the AT customer table has fewer null values in the fields used by the current identity resolution process. *Proposed approach*: If an SP owner record contains incomplete information (e.g., missing first name or last name), we could leverage the purchaser's information as a fallback source to support identity matching.